# **Four Core Concepts — Coding Exercise**

**Learning Objective**

This exercise practices the four core concepts from the lesson: **abstraction**, **encapsulation**, **inheritance**, and **polymorphism** (including composition). Each task builds on the previous one, so complete them in order and keep all your classes in the same notebook.

## **Laboratory Exercise: Campus Sensor Monitor**

### **Scenario**

The university is installing sensors in campus buildings to monitor the environment. Each sensor has an **ID** and a **location**, and it stores a list of **readings**.

There are different kinds of sensors:

| Sensor type   | Unit | Valid readings     |
| ------------- | ---- | ------------------ |
| Temperature   | °C   | −40 to 60          |
| Humidity      | %    | 0 to 100           |

You will build the classes for this system step by step. By the end, your design should look like this:

```{mermaid}
classDiagram
class Sensor {
    <<abstract>>
    -String _sensor_id
    -String _location
    -list _readings
    +sensor_id
    +readings
    +unit()* str
    +is_valid(value)* bool
    +record(value)
    +average() float
    +describe() str
}
class TemperatureSensor {
    +unit() str
    +is_valid(value) bool
}
class HumiditySensor {
    +unit() str
    +is_valid(value) bool
}
class SmartTemperatureSensor {
    +float alert_level
    +describe() str
}
class Building {
    +String name
    -list _sensors
    +add_sensor(sensor)
    +report()
}
Sensor <|-- TemperatureSensor
Sensor <|-- HumiditySensor
TemperatureSensor <|-- SmartTemperatureSensor
Building o-- Sensor : monitors
```

### **Task 1 — Abstraction: The `Sensor` Base Class**

Create an **abstract base class** named `Sensor` using the `abc` module.

```python
from abc import ABC, abstractmethod
```

The constructor should accept:

* `sensor_id`
* `location`

and start with an **empty list** of readings.

Declare two **abstract methods**:

| Method            | Purpose                                              |
| ----------------- | ---------------------------------------------------- |
| `unit()`          | returns the unit of measurement as a string          |
| `is_valid(value)` | returns `True` if the value is an acceptable reading |

Then try to create a `Sensor` object directly:

```python
s = Sensor("X-01", "Lobby")
```

Use `try` / `except` to catch the error and print it.

Answer in a Markdown cell:

> Why does Python refuse to create a `Sensor` object?

### **Task 2 — Encapsulation: Protecting the Readings**

Inside `Sensor`, store the data in **protected attributes**:

```python
self._sensor_id
self._location
self._readings
```

Add the following members to `Sensor`:

| Member            | Kind               | Behavior                                                                 |
| ----------------- | ------------------ | ------------------------------------------------------------------------ |
| `sensor_id`       | read-only property | returns the sensor's ID                                                  |
| `location`        | read-only property | returns the sensor's location                                            |
| `readings`        | read-only property | returns the readings as a **tuple** (so the caller cannot modify the list) |
| `record(value)`   | method             | adds the value **only if** `self.is_valid(value)` is `True`; otherwise raises `ValueError` |
| `average()`       | method             | returns the mean of the readings, or `None` if there are no readings     |

The error message for an invalid reading should look like:

```text
ValueError: 75 is not a valid reading for TEMP-01
```

Answer in a Markdown cell:

> Why does `readings` return a **tuple** instead of the list itself?

### **Task 3 — Inheritance: Concrete Sensors**

Create two subclasses of `Sensor`:

```python
class TemperatureSensor(Sensor):
    ...

class HumiditySensor(Sensor):
    ...
```

Each subclass must implement `unit()` and `is_valid(value)` using the table in the scenario.

Test your classes:

```python
temp = TemperatureSensor("TEMP-01", "Library")
temp.record(28.5)
temp.record(30.0)
temp.record(31.5)

hum = HumiditySensor("HUM-01", "Library")
hum.record(65)
hum.record(70)
hum.record(72)

print(temp.average(), temp.unit())
print(hum.average(), hum.unit())
print(temp.readings)
```

Expected output:

```text
30.0 °C
69.0 %
(28.5, 30.0, 31.5)
```

Then show that validation works:

```python
temp.record(75)    # should raise ValueError
hum.record(-5)     # should raise ValueError
```

Use `try` / `except` so your notebook continues running.

### **Task 4 — A Shared `describe()` Method**

Add a **concrete** method `describe()` to the `Sensor` base class. Because it is defined in the parent, **both** subclasses inherit it automatically.

It should return a string in this format:

```text
TEMP-01 @ Library: avg 30.00 °C (3 readings)
HUM-01 @ Library: avg 69.00 % (3 readings)
```

If the sensor has no readings yet, it should return:

```text
TEMP-03 @ Canteen: no readings yet
```

Notice that `describe()` calls `self.unit()`, which is different for each subclass — this is your first taste of polymorphism.

### **Task 5 — Method Overriding and `super()`**

The server room needs a temperature sensor that raises an alert when the average temperature is too high.

Create:

```python
class SmartTemperatureSensor(TemperatureSensor):
    def __init__(self, sensor_id, location, alert_level):
        ...
```

Requirements:

1. Use `super().__init__(...)` to reuse the parent constructor.
2. Store `alert_level` as an attribute.
3. **Override** `describe()`. Call `super().describe()` and, if the average is **above** `alert_level`, append an alert message.

Test:

```python
smart = SmartTemperatureSensor("TEMP-02", "Server Room", alert_level=27)
smart.record(26.0)
smart.record(28.0)
smart.record(29.0)

print(smart.describe())
```

Expected output:

```text
TEMP-02 @ Server Room: avg 27.67 °C (3 readings) | ALERT: above 27 °C
```

Finally, print:

```python
print(isinstance(smart, TemperatureSensor))
print(isinstance(smart, Sensor))
print([cls.__name__ for cls in SmartTemperatureSensor.__mro__])
```

### **Task 6 — Polymorphism and Duck Typing**

Put all your sensors in one list and display them with a **single loop**:

```python
sensors = [temp, hum, smart, TemperatureSensor("TEMP-03", "Canteen")]

for sensor in sensors:
    print(sensor.describe())
```

Expected output:

```text
TEMP-01 @ Library: avg 30.00 °C (3 readings)
HUM-01 @ Library: avg 69.00 % (3 readings)
TEMP-02 @ Server Room: avg 27.67 °C (3 readings) | ALERT: above 27 °C
TEMP-03 @ Canteen: no readings yet
```

Do **not** use `isinstance()` or `type()` inside the loop.

Next, create a class `ManualLog` that does **not** inherit from `Sensor`, but has its own `describe()` method:

```python
log = ManualLog("Guard checked the server room at 9 PM")
print(log.describe())
```

```text
Manual log: Guard checked the server room at 9 PM
```

Add `log` to the `sensors` list and run the loop again.

Answer in a Markdown cell:

> `ManualLog` is not a `Sensor`, yet the loop still works. What is this behavior called, and why does it work?

### **Task 7 — Composition: The `Building` Class**

A building **has** sensors; it is **not** a sensor. Model this using composition.

Create a class `Building` with:

| Member               | Behavior                                                    |
| -------------------- | ----------------------------------------------------------- |
| `name`               | the building's name                                         |
| `_sensors`           | a protected list of monitored objects                       |
| `add_sensor(sensor)` | adds a sensor (or any object with `describe()`) to the list |
| `report()`           | prints a header and then the `describe()` of every sensor   |

Test:

```python
ict = Building("ICT Building")
ict.add_sensor(temp)
ict.add_sensor(hum)
ict.add_sensor(smart)
ict.report()
```

Expected output:

```text
=== ICT Building (3 sensors) ===
TEMP-01 @ Library: avg 30.00 °C (3 readings)
HUM-01 @ Library: avg 69.00 % (3 readings)
TEMP-02 @ Server Room: avg 27.67 °C (3 readings) | ALERT: above 27 °C
```

Answer in a Markdown cell:

> Why is `Building` designed using **composition** instead of making `Building` a subclass of `Sensor`?

### **Task 8 — Reflection**

In a Markdown cell, identify **where** each core concept appears in your solution. Complete the table:

| Concept        | Where it appears in your code |
| -------------- | ----------------------------- |
| Abstraction    |                               |
| Encapsulation  |                               |
| Inheritance    |                               |
| Polymorphism   |                               |
| Composition    |                               |

**Challenge (optional):** Add a third sensor type, such as `AirQualitySensor` (unit `AQI`, valid readings 0 to 500). How many lines of the `Building` class or the Task 6 loop did you need to change? Explain why.

## **Submission Requirements**

Submit **one Jupyter Notebook** containing all tasks in order. Each task should have:

* the code that solves it;
* the output produced by running the code; and
* the Markdown answers to the questions asked.

Make sure the notebook runs from top to bottom without errors (**Kernel → Restart & Run All**) before submitting.